# grabs the customers from customer-orders.csv and shows the total

In [11]:
from pyspark.sql import SparkSession, functions as func
from pyspark.sql.types import StructType, StructField, FloatType, IntegerType


spark = SparkSession.builder.appName("test").getOrCreate()


schema = StructType([
    StructField('customerId', IntegerType()),
    StructField('orderId', IntegerType()),
    StructField('total', FloatType()),
])

customer = spark.read.csv('../../customer-orders.csv', schema=schema, header=False)

# have to specify agg so we can make an alias for the total amount a customer spent
total = customer.groupBy('customerId').agg(func.sum('total').alias("total_spent"))



total.orderBy('total_spent', ascending = False).show()

spark.stop()



+----------+------------------+
|customerId|       total_spent|
+----------+------------------+
|        68| 6375.450028181076|
|        73| 6206.199985742569|
|        39| 6193.109993815422|
|        54| 6065.390002984554|
|        71| 5995.659991919994|
|         2| 5994.589979887009|
|        97| 5977.190007060766|
|        46| 5963.110011339188|
|        42| 5696.840004444122|
|        59| 5642.890004396439|
|        41| 5637.619991332293|
|         0| 5524.950008839369|
|         8|5517.2399980425835|
|        85|  5503.42998456955|
|        61| 5497.479998707771|
|        32| 5496.049998283386|
|        58| 5437.730004191399|
|        63| 5415.150004655123|
|        15| 5413.510010659695|
|         6| 5397.880012750626|
+----------+------------------+
only showing top 20 rows


In [12]:
# using spark sql instead of using dataframe to do the groupby and agg

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, FloatType, IntegerType


spark = SparkSession.builder.appName("sql").getOrCreate()

schema = StructType([
    StructField('customerId', IntegerType()),
    StructField('orderId', IntegerType()),
    StructField('total', FloatType()),
])


customer = spark.read.csv('../../customer-orders.csv', schema=schema, header=False)

customer.createOrReplaceTempView('orders')

customers = spark.sql("""SELECT customerId, SUM(total) AS total_spent FROM orders GROUP BY customerId ORDER BY total_spent DESC
""")

customers.show()
spark.stop()

+----------+------------------+
|customerId|       total_spent|
+----------+------------------+
|        68| 6375.450028181076|
|        73| 6206.199985742569|
|        39| 6193.109993815422|
|        54| 6065.390002984554|
|        71| 5995.659991919994|
|         2| 5994.589979887009|
|        97| 5977.190007060766|
|        46| 5963.110011339188|
|        42| 5696.840004444122|
|        59| 5642.890004396439|
|        41| 5637.619991332293|
|         0| 5524.950008839369|
|         8|5517.2399980425835|
|        85|  5503.42998456955|
|        61| 5497.479998707771|
|        32| 5496.049998283386|
|        58| 5437.730004191399|
|        63| 5415.150004655123|
|        15| 5413.510010659695|
|         6| 5397.880012750626|
+----------+------------------+
only showing top 20 rows


# get log info challenge

In [8]:
from pyspark.sql import SparkSession, functions as func
from pyspark.sql.types import StructType, StructField, FloatType, IntegerType, StringType


spark = SparkSession.builder.appName("app").getOrCreate()
df = spark.read.text("../../access_log.txt", lineSep="\n")
df2 = df.withColumn("parts", func.split("value", " "))
df3 = (
    df2.withColumn("field0", func.col("parts")[0])
       .withColumn("field1", func.col("parts")[1])
       .withColumn("field2", func.col("parts")[2])
        .withColumn("field3", func.col("parts")[3])
       .withColumn("field4", func.col("parts")[4])
       .withColumn("field5", func.col("parts")[5])
       .withColumn("field6", func.col("parts")[6])
        .withColumn("field7", func.col("parts")[7])
       .withColumn("field8", func.col("parts")[8])
       .withColumn("field9", func.col("parts")[9])
)
df3 = df3.drop("value")
df3 = df3.drop("parts")
df3.show()

print(df.head())
df.createOrReplaceTempView('logs_table')



+--------------+------+------+--------------------+------+------+--------------------+---------+------+------+
|        field0|field1|field2|              field3|field4|field5|              field6|   field7|field8|field9|
+--------------+------+------+--------------------+------+------+--------------------+---------+------+------+
| 66.249.75.159|     -|     -|[29/Nov/2015:03:5...|+0000]|  "GET|         /robots.txt|HTTP/1.1"|   200|    55|
| 66.249.75.168|     -|     -|[29/Nov/2015:03:5...|+0000]|  "GET|              /blog/|HTTP/1.1"|   200|  8083|
|185.71.216.232|     -|     -|[29/Nov/2015:03:5...|+0000]| "POST|       /wp-login.php|HTTP/1.1"|   200|  1691|
|54.165.199.171|     -|     -|[29/Nov/2015:04:3...|+0000]|  "GET|  /sitemap_index.xml|HTTP/1.0"|   200|   592|
|54.165.199.171|     -|     -|[29/Nov/2015:04:3...|+0000]|  "GET|   /post-sitemap.xml|HTTP/1.0"|   200|  2502|
|54.165.199.171|     -|     -|[29/Nov/2015:04:3...|+0000]|  "GET|   /page-sitemap.xml|HTTP/1.0"|   200| 11462|
|

# finding the top 10 most popular movies in u.data

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, FloatType, IntegerType, StringType


spark = SparkSession.builder.appName("log").getOrCreate()

schema = StructType([
    StructField('userId', IntegerType()),
    StructField('movieId', IntegerType()),
    StructField('rating', IntegerType()),
    StructField('timestamp', IntegerType())
])

df = spark.read.option("sep", "\t").schema(schema).csv("../../ml-100k/u.data")

df.sort('rating', ascending=False).show(10)



+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|  NULL|   NULL|  NULL|     NULL|
|  NULL|   NULL|  NULL|     NULL|
|  NULL|   NULL|  NULL|     NULL|
|  NULL|   NULL|  NULL|     NULL|
|  NULL|   NULL|  NULL|     NULL|
|  NULL|   NULL|  NULL|     NULL|
|  NULL|   NULL|  NULL|     NULL|
|  NULL|   NULL|  NULL|     NULL|
|  NULL|   NULL|  NULL|     NULL|
|  NULL|   NULL|  NULL|     NULL|
+------+-------+------+---------+
only showing top 10 rows
